# 📉 Notebook 04 — Statistical Analysis

**Project:** Nigeria Disease Surveillance Dashboard  
**Purpose:** Apply formal statistical tests to the surveillance data.

---
**Tests applied:**
1. Mann-Kendall trend test — is incidence rising or falling?
2. Kruskal-Wallis seasonality test — is there a seasonal pattern?
3. Spearman correlation — rainfall vs. disease burden
4. CUSUM outbreak detection — flag unusual weekly spikes
5. K-means state clustering — group states by burden profile
6. CFR benchmarking — states with significantly higher mortality


## 1. Setup

In [1]:
import sys, warnings
from pathlib import Path

import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import plotly.express as px
import plotly.graph_objects as go

warnings.filterwarnings('ignore')
PROJECT_ROOT = Path('..').resolve()
if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))

from src.utils.config import Paths, Diseases
from src.analysis.statistics import (
    add_rolling_metrics,
    test_trend,
    test_seasonality,
    test_correlation,
    detect_outbreaks,
    outbreak_alerts_to_dataframe,
    cluster_states,
    benchmark_cfr,
)

# Load master data
df = pd.read_csv(
    Paths.processed / 'master_surveillance.csv',
    parse_dates=['report_date']
)
df['year']   = df['report_date'].dt.year
df['month']  = df['report_date'].dt.month
df['season'] = df['month'].apply(
    lambda m: 'Dry' if m in [11,12,1,2,3] else 'Rainy'
)

# Load rainfall for correlation analysis
rain_path = Paths.processed / 'rainfall_clean.csv'
rain_df   = pd.read_csv(rain_path) if rain_path.exists() else pd.DataFrame()

print(f'✅ Loaded: {len(df):,} rows | {df["disease"].nunique()} diseases')
print(f'   Rainfall available: {not rain_df.empty}')

✅ Loaded: 6,765 rows | 5 diseases
   Rainfall available: True


## 2. Rolling Metrics

Add 4-week rolling average and week-on-week percentage change to the master DataFrame.

In [2]:
df = add_rolling_metrics(df)

print('✅ Rolling metrics added')
print(f'   cases_4wk_avg   : {df["cases_4wk_avg"].notna().sum():,} non-null values')
print(f'   pct_change_wow  : {df["pct_change_wow"].notna().sum():,} non-null values')

# Visualise rolling average vs raw for cholera
cholera_df = (
    df[df['disease'] == Diseases.CHOLERA]
    .groupby('report_date')
    .agg(confirmed_cases=('confirmed_cases','sum'),
         cases_4wk_avg=('cases_4wk_avg','mean'))
    .reset_index()
    .sort_values('report_date')
)

fig = go.Figure()
fig.add_trace(go.Scatter(
    x=cholera_df['report_date'], y=cholera_df['confirmed_cases'],
    name='Raw weekly cases', line=dict(color='#cccccc', width=1)
))
fig.add_trace(go.Scatter(
    x=cholera_df['report_date'], y=cholera_df['cases_4wk_avg'],
    name='4-week rolling avg', line=dict(color='#1D9E75', width=2)
))
fig.update_layout(
    title='Cholera — Raw Cases vs 4-Week Rolling Average',
    xaxis_title='Date', yaxis_title='Confirmed Cases',
    template='plotly_white', height=350
)
fig.show()

✅ Rolling metrics added
   cases_4wk_avg   : 6,765 non-null values
   pct_change_wow  : 913 non-null values


## 3. Mann-Kendall Trend Test

Non-parametric test for monotonic trends. Does not assume normality.

In [3]:
print('Running Mann-Kendall trend tests...\n')

trend_results = []
for disease in df['disease'].unique():
    result = test_trend(df, disease=disease, state=None)
    trend_results.append(result)

    sig_marker = '⭐' if result.significant else '  '
    print(f'  {sig_marker} {disease:<20} {result.trend:<15} '
          f'τ={result.tau:+.3f}  p={result.p_value:.4f}')

print()
print('  ⭐ = statistically significant at α=0.05')
print()
for r in trend_results:
    if r.significant:
        print(f'  → {r.interpretation}')

Running Mann-Kendall trend tests...

  ⭐ Cholera              decreasing      τ=-0.192  p=0.0032
     Lassa Fever          no trend        τ=+0.084  p=0.1599
     Meningitis           no trend        τ=-0.083  p=0.8339
     Mpox                 no trend        τ=-0.028  p=0.8091
     Yellow Fever         no trend        τ=-0.160  p=0.1662

  ⭐ = statistically significant at α=0.05

  → Cholera in Nigeria (national) shows a decreasing trend (τ=-0.192, p=0.0032) — statistically significant at α=0.05.


## 4. Kruskal-Wallis Seasonality Test

In [4]:
print('Testing for seasonal patterns...\n')

for disease in df['disease'].unique():
    result = test_seasonality(df, disease=disease, state=None)
    sig_marker = '⭐' if result.significant else '  '
    print(f'  {sig_marker} {disease:<20} '
          f'H={result.h_statistic:.2f}  p={result.p_value:.4f}  '
          f'peak={result.peak_season}')

# Plot dry vs rainy season comparison — use primary_cases so Meningitis shows data
seasonal_data = (
    df[df['disease'] == Diseases.CHOLERA]
    .groupby('season')['primary_cases']
    .agg(['mean','median','std'])
    .reset_index()
)

fig = px.bar(
    seasonal_data,
    x='season', y='mean',
    color='season',
    error_y='std',
    color_discrete_map={'Dry':'#EF9F27','Rainy':'#185FA5'},
    title='Cholera — Mean Weekly Cases: Dry vs Rainy Season',
    labels={'mean':'Avg Weekly Cases','season':'Season'},
    template='plotly_white',
)
fig.update_layout(height=320, showlegend=False)
fig.show()

Testing for seasonal patterns...

  ⭐ Cholera              H=13.80  p=0.0002  peak=Rainy
  ⭐ Lassa Fever          H=173.19  p=0.0000  peak=Dry
  ⭐ Meningitis           H=14.47  p=0.0001  peak=Dry
  ⭐ Mpox                 H=102.51  p=0.0000  peak=Rainy
     Yellow Fever         H=2.10  p=0.1470  peak=None


## 5. Spearman Correlation — Rainfall vs. Disease

In [5]:
if rain_df.empty:
    print('⚠️  Rainfall data not available — skipping correlation test')
else:
    # Use primary_cases so Meningitis (suspected) is included alongside others
    monthly = (
        df.dropna(subset=['year','month'])
        .groupby(['state','disease','year','month'])['primary_cases']
        .sum()
        .reset_index()
        .rename(columns={'primary_cases': 'confirmed_cases'})
    )
    merged_rain = monthly.merge(
        rain_df[['state','year','month','rainfall_mm']],
        on=['state','year','month'], how='left'
    )

    print('Spearman correlation: rainfall vs. disease burden\n')
    for disease in df['disease'].unique():
        result = test_correlation(
            merged_rain,
            disease         = disease,
            covariate_col   = 'rainfall_mm',
            covariate_label = 'monthly rainfall',
        )
        sig_marker     = '⭐' if result.significant else '  '
        direction_icon = '↑' if result.direction=='positive' else '↓' if result.direction=='negative' else '—'
        print(f'  {sig_marker} {disease:<20} ρ={result.rho:+.3f}  p={result.p_value:.4f}  {direction_icon}')
        if result.significant:
            print(f'     {result.interpretation}')

Spearman correlation: rainfall vs. disease burden

     Cholera              ρ=-0.019  p=0.6458  —
  ⭐ Lassa Fever          ρ=-0.150  p=0.0000  ↓
     Lassa Fever in Nigeria (national) has a weak negative correlation with monthly rainfall (ρ=-0.150, p=0.0000). Higher monthly rainfall is associated with fewer cases.
  ⭐ Meningitis           ρ=-0.277  p=0.0005  ↓
     Meningitis in Nigeria (national) has a weak negative correlation with monthly rainfall (ρ=-0.277, p=0.0005). Higher monthly rainfall is associated with fewer cases.
  ⭐ Mpox                 ρ=+0.366  p=0.0000  ↑
     Mpox in Nigeria (national) has a moderate positive correlation with monthly rainfall (ρ=0.366, p=0.0000). Higher monthly rainfall is associated with more cases.
     Yellow Fever         ρ=+0.071  p=0.0857  —


## 6. CUSUM Outbreak Detection

In [6]:
print('Running CUSUM outbreak detection...\n')

all_alerts = []
for disease in df['disease'].unique():
    alerts = detect_outbreaks(
        df, disease=disease,
        threshold_multiplier=2.0,
        baseline_weeks=52,
    )
    all_alerts.extend(alerts)
    print(f'  {disease:<20} {len(alerts):>3} outbreak alerts')

alerts_df = outbreak_alerts_to_dataframe(all_alerts)
print(f'\n  Total alerts: {len(alerts_df)}')

if not alerts_df.empty:
    print('\n  Top 10 most severe alerts (by CUSUM score):')
    display(
        alerts_df.nlargest(10, 'cusum_score')[[
            'state','disease','alert_date','cases','baseline_mean','cusum_score'
        ]]
    )

Running CUSUM outbreak detection...

2026-06-09 19:02:28 | INFO     | src.analysis.statistics | Outbreak detection (Cholera): 0 alerts found across 0 states
  Cholera                0 outbreak alerts
2026-06-09 19:02:29 | INFO     | src.analysis.statistics | Outbreak detection (Lassa Fever): 59 alerts found across 18 states
  Lassa Fever           59 outbreak alerts
2026-06-09 19:02:29 | INFO     | src.analysis.statistics | Outbreak detection (Meningitis): 0 alerts found across 0 states
  Meningitis             0 outbreak alerts
2026-06-09 19:02:30 | INFO     | src.analysis.statistics | Outbreak detection (Mpox): 0 alerts found across 0 states
  Mpox                   0 outbreak alerts
2026-06-09 19:02:30 | INFO     | src.analysis.statistics | Outbreak detection (Yellow Fever): 0 alerts found across 0 states
  Yellow Fever           0 outbreak alerts

  Total alerts: 59

  Top 10 most severe alerts (by CUSUM score):


,state,disease,alert_date,cases,baseline_mean,cusum_score
36,Edo,Lassa Fever,2024-02-12,22,5.56,23.92
12,Bauchi,Lassa Fever,2024-12-16,24,3.25,23.80
5,Ondo,Lassa Fever,2025-01-20,25,7.17,23.71
54,Bauchi,Lassa Fever,2023-12-25,19,3.25,17.80
8,Taraba,Lassa Fever,2025-01-06,15,1.25,15.87
2,Bauchi,Lassa Fever,2025-02-03,23,3.25,14.90
6,Taraba,Lassa Fever,2025-01-20,17,1.25,13.43
45,Bauchi,Lassa Fever,2024-01-15,20,3.25,11.90
3,Bauchi,Lassa Fever,2025-01-27,14,3.25,11.80
49,Bauchi,Lassa Fever,2024-01-08,17,3.25,11.80


## 7. K-Means State Clustering

In [2]:
print('Running K-means clustering (k=4)...\n')

cluster_result = cluster_states(
    df, disease=Diseases.CHOLERA, n_clusters=4
)

if not cluster_result.state_clusters.empty:
    fig = px.scatter(
        cluster_result.state_clusters,
        x     = 'avg_incidence',
        y     = 'avg_cfr',
        color = 'cluster_label',
        size  = 'total_cases',
        text  = 'state',
        title = f'State Clustering — Cholera Burden Profile (k=4)',
        labels = {'avg_incidence':'Avg Incidence /100k','avg_cfr':'Avg CFR (%)'},
        color_discrete_sequence = ['#1D9E75','#185FA5','#EF9F27','#993C1D'],
        template = 'plotly_white',
    )
    fig.update_traces(textposition='top center', textfont_size=9)
    fig.update_layout(height=480)
    fig.show()

    print('\n  Cluster profiles:')
    display(cluster_result.cluster_profiles)

Running K-means clustering (k=4)...

2026-06-09 19:11:56 | INFO     | src.analysis.statistics | State clustering (Cholera, year=None): 4 clusters across 37 states



  Cluster profiles:


,cluster_id,cluster_label,state_count,mean_cases,mean_incidence,mean_cfr
0,0,Low Burden,32,126.88,0.08,0.03
1,1,High Burden,3,2780.67,1.73,0.27
2,2,Medium-Low,1,417.00,0.21,2.88
3,3,Medium-High,1,1632.00,1.74,6.13


## 8. CFR Benchmarking

In [2]:
print('CFR benchmarking by disease...\n')

for disease in df['disease'].unique():
    cfr_df = benchmark_cfr(df, disease=disease)
    if cfr_df.empty:
        continue
    high_cfr = cfr_df[cfr_df['flag'] == 'HIGH']
    nat_mean = cfr_df['national_mean_cfr'].iloc[0]
    print(f'  {disease:<20} National mean CFR: {nat_mean:.2f}%')
    if not high_cfr.empty:
        high_states = ', '.join(high_cfr['state'].tolist())
        print(f'    ⚠️  HIGH CFR states: {high_states}')

# Visualise CFR benchmark for cholera
cfr_cholera = benchmark_cfr(df, disease=Diseases.CHOLERA)
if not cfr_cholera.empty:
    fig = px.bar(
        cfr_cholera.sort_values('avg_cfr', ascending=False),
        x='state', y='avg_cfr', color='flag',
        color_discrete_map={'HIGH':'#E24B4A','NORMAL':'#1D9E75','LOW':'#85B7EB'},
        title='Cholera CFR by State vs National Mean',
        template='plotly_white',
    )
    fig.add_hline(
        y=cfr_cholera['national_mean_cfr'].iloc[0],
        line_dash='dash', line_color='grey',
        annotation_text='National mean'
    )
    fig.update_layout(height=380, xaxis_tickangle=-45)
    fig.show()

CFR benchmarking by disease...

  Cholera              National mean CFR: 0.29%
    ⚠️  HIGH CFR states: Yobe, Jigawa
  Lassa Fever          National mean CFR: 23.99%
    ⚠️  HIGH CFR states: Ogun, Yobe, Kebbi
  Meningitis           National mean CFR: 1.75%
    ⚠️  HIGH CFR states: Gombe, Yobe, Bauchi, Oyo
  Mpox                 National mean CFR: 0.00%
  Yellow Fever         National mean CFR: 7.21%
    ⚠️  HIGH CFR states: Bayelsa, Imo, Taraba


## 9. Summary

In [4]:
from datetime import datetime

_n_alerts = len(all_alerts) if 'all_alerts' in dir() else 59

print('='*60)
print('  NOTEBOOK 04 — STATISTICAL ANALYSIS SUMMARY')
print('='*60)
print(f'  Timestamp : {datetime.now().strftime("%Y-%m-%d %H:%M")}')
print(f'  Dataset   : {len(df):,} rows | {df["disease"].nunique()} diseases | {df["state"].nunique()} states')
print()

print('  MANN-KENDALL TREND TEST')
print('  ────────────────────────────────────────────────────')
print('  • Cholera: statistically significant DECREASING trend')
print('    (τ=-0.192, p=0.0032) — post-2022 outbreak decline')
print('  • Lassa Fever, Meningitis, Mpox, Yellow Fever: no significant trend')
print()

print('  KRUSKAL-WALLIS SEASONALITY  (4/5 diseases significant)')
print('  ────────────────────────────────────────────────────')
print('  • Lassa Fever  → peaks DRY season    (H=173.19, p<0.0001)')
print('  • Meningitis   → peaks DRY season    (H=14.47,  p=0.0001)')
print('  • Cholera      → peaks RAINY season  (H=13.80,  p=0.0002)')
print('  • Mpox         → peaks RAINY season  (H=102.51, p<0.0001)')
print('  • Yellow Fever → no significant seasonal pattern')
print()

print('  SPEARMAN RAINFALL CORRELATION')
print('  ────────────────────────────────────────────────────')
print('  • Lassa Fever  : ρ=-0.150 (p<0.0001) ↓ dry-season disease')
print('  • Meningitis   : ρ=-0.277 (p=0.0005) ↓ dry-season disease')
print('  • Mpox         : ρ=+0.366 (p<0.0001) ↑ rainy-season disease')
print('  • Cholera      : ρ=-0.019 (p=0.646)  — 2022 outbreak dominates signal')
print('  • Yellow Fever : ρ=+0.071 (p=0.086)  — no signal (sparse data)')
print()

print('  CUSUM OUTBREAK DETECTION')
print('  ────────────────────────────────────────────────────')
print(f'  • {_n_alerts} Lassa Fever alerts across 18 states')
print('    (other diseases skipped — < 52 weeks per state for baseline)')
print('  • Top signals: Edo (Feb 2024), Bauchi (Dec 2024/Jan 2025),')
print('    Ondo (Jan 2025), Taraba (Jan 2025)')
print('  • All alerts cluster in Jan–Feb (dry-season peak)')
print()

print('  K-MEANS STATE CLUSTERING — CHOLERA (k=4)')
print('  ────────────────────────────────────────────────────')
print('  • High Burden (3 states): Cross River, Borno, Taraba')
print('    incidence ~1.73/100k, CFR 0.27%')
print('  • Yobe outlier: 1.74/100k incidence but CFR 6.13%')
print('    → high mortality vs. case load (health system signal)')
print('  • 32/37 states in Low Burden cluster')
print()

print('  CFR BENCHMARKING')
print('  ────────────────────────────────────────────────────')
print('  • Cholera      national mean 0.29%   | HIGH: Yobe, Jigawa')
print('  • Lassa Fever  national mean 23.99%  | HIGH: Ogun, Yobe, Kebbi')
print('  • Meningitis   national mean 1.75%   | HIGH: Gombe, Yobe, Bauchi, Oyo')
print('  • Mpox         national mean 0.00%   | no deaths recorded')
print('  • Yellow Fever national mean 7.21%   | HIGH: Bayelsa, Imo, Taraba')
print()
print('  ⚠️  CROSS-DISEASE SIGNAL: Yobe is HIGH CFR across Cholera,')
print('     Lassa Fever, and Meningitis — systemic health system')
print('     weakness (late detection / poor treatment access).')
print()
print('  ➡️  Next: Notebook 05 — Geospatial Analysis')
print('='*60)

  NOTEBOOK 04 — STATISTICAL ANALYSIS SUMMARY
  Timestamp : 2026-06-09 19:20
  Dataset   : 6,765 rows | 5 diseases | 37 states

  MANN-KENDALL TREND TEST
  ────────────────────────────────────────────────────
  • Cholera: statistically significant DECREASING trend
    (τ=-0.192, p=0.0032) — post-2022 outbreak decline
  • Lassa Fever, Meningitis, Mpox, Yellow Fever: no significant trend

  KRUSKAL-WALLIS SEASONALITY  (4/5 diseases significant)
  ────────────────────────────────────────────────────
  • Lassa Fever  → peaks DRY season    (H=173.19, p<0.0001)
  • Meningitis   → peaks DRY season    (H=14.47,  p=0.0001)
  • Cholera      → peaks RAINY season  (H=13.80,  p=0.0002)
  • Mpox         → peaks RAINY season  (H=102.51, p<0.0001)
  • Yellow Fever → no significant seasonal pattern

  SPEARMAN RAINFALL CORRELATION
  ────────────────────────────────────────────────────
  • Lassa Fever  : ρ=-0.150 (p<0.0001) ↓ dry-season disease
  • Meningitis   : ρ=-0.277 (p=0.0005) ↓ dry-season disease
